# PyMolFit 05: Diagnostics and troubleshooting

This notebook compares an intentionally incomplete O2 B-band fit with a better instrumental and wavelength model. It then builds a health report from optimizer state, parameter bounds, transmission depth, masks, and residual structure.

No single scalar proves that a telluric correction is scientifically correct. Diagnostics identify suspicious behaviour; external validation, repeat observations, standards, injection tests, or comparison with an independent method are still required for publication-quality claims.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table
from pymolfit import correction_summary, correct_file

candidates = (Path.cwd() / "tutorials", Path.cwd())
TUTORIAL_ROOT = next((p.resolve() for p in candidates if (p / "data").is_dir()), None)
if TUTORIAL_ROOT is None:
    raise FileNotFoundError("Could not locate tutorials/data")
INPUT = TUTORIAL_ROOT / "data" / "harps_o2_bband_crop_air.fits"
OUTPUT = TUTORIAL_ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)
FIT_RANGE = ((0.6868, 0.6905),)
PLOT_RANGE_ANGSTROM = (6868.0, 6905.0)

## An intentionally incomplete fit

This baseline has no instrumental convolution and no fitted wavelength displacement. It can still terminate successfully, demonstrating why optimizer success alone is insufficient. O2 is selected explicitly because this predetermined interval is the O2 B band.

In [ ]:
incomplete = correct_file(
    input_path=INPUT,
    wavelength_medium="air",
    aer_catalog="auto",
    hitran_species=("O2",),
    gdas_mode="average",
    fit_ranges=FIT_RANGE,
    continuum_order=1,
    solve_continuum_linear=True,
)
print("Success:", incomplete.success)
print("Message:", incomplete.message)
print("Species scales:", incomplete.species_scales)

## Add an instrumental profile and wavelength alignment

HARPS has approximately one-Angstrom-per-million wavelength sampling in this product. At the O2 B band, its nominal resolving power implies a Gaussian starting width near 2.5 pixels. The width and shift are fitted rather than fixed. A quadratic continuum allows smooth local source and throughput curvature without following individual atmospheric lines.

In [ ]:
improved = correct_file(
    input_path=INPUT,
    output_path=OUTPUT / "05_o2_corrected.ecsv",
    product_path=OUTPUT / "05_o2_fit_product.ecsv",
    wavelength_medium="air",
    aer_catalog="auto",
    hitran_species=("O2",),
    gdas_mode="average",
    fit_ranges=FIT_RANGE,
    continuum_order=2,
    solve_continuum_linear=True,
    lsf_box_width_pixels=1.0,
    lsf_sigma_pixels=2.5,
    fit_lsf_sigma=True,
    lsf_sigma_bounds=(0.5, 4.5),
    lsf_lorentz_fwhm_pixels=0.5,
    fit_lsf_lorentz_fwhm=True,
    lsf_lorentz_fwhm_bounds=(0.0, 6.0),
    fit_wavelength_shift=True,
    wavelength_shift_bounds=(-2e-5, 2e-5),
)
print("Success:", improved.success)
print("Message:", improved.message)
print("Species scales:", improved.species_scales)
print("Parameters at bounds:", improved.parameter_bound_status or "none")

## Inspect both forward models

Paired positive and negative residuals around line centres usually indicate wavelength or LSF mismatch. Residuals with the same sign at line centres suggest incorrect atmospheric depth. Broad residual curvature indicates an inadequate continuum or unmasked source structure.

In [ ]:
def plot_diagnostics(result, title):
    wave = result.spectrum.to_air().to_unit("angstrom").wavelength
    flux = np.asarray(result.spectrum.flux, dtype=float)
    continuum = np.asarray(result.continuum, dtype=float)
    model = np.asarray(result.model_flux, dtype=float)
    corrected = np.asarray(result.corrected.flux, dtype=float)
    transmission = np.asarray(result.transmission, dtype=float)
    visible = (wave >= PLOT_RANGE_ANGSTROM[0]) & (wave <= PLOT_RANGE_ANGSTROM[1])
    scale = np.nanmedian(flux[visible])
    residual = np.divide(flux - model, continuum, out=np.full_like(flux, np.nan), where=continuum != 0)

    figure, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
    axes[0].plot(wave, flux / scale, color="black", label="Observed")
    axes[0].plot(wave, model / scale, color="tab:orange", label="Forward model")
    axes[0].set_ylabel("Normalized flux")
    axes[0].legend()
    axes[1].plot(wave, transmission, color="tab:blue")
    axes[1].set_ylabel("Transmission")
    axes[2].plot(wave, corrected / scale, color="tab:green", label="Corrected")
    axes[2].plot(wave, continuum / scale, color="0.4", linestyle="--", label="Continuum")
    axes[2].set_ylabel("Normalized flux")
    axes[2].legend()
    axes[3].plot(wave, residual, color="tab:purple", linewidth=0.9)
    axes[3].axhline(0, color="black", linewidth=0.8)
    axes[3].set_ylabel("Residual / continuum")
    axes[3].set_xlabel("Air wavelength [Angstrom]")
    for axis in axes:
        axis.set_xlim(*PLOT_RANGE_ANGSTROM)
        axis.grid(alpha=0.15)
    figure.suptitle(title)
    figure.tight_layout()
    plt.show()

plot_diagnostics(incomplete, "Incomplete O2 fit")
plot_diagnostics(improved, "Improved O2 fit")

In [ ]:
wave = improved.spectrum.to_air().to_unit("angstrom").wavelength
keep = (wave >= PLOT_RANGE_ANGSTROM[0]) & (wave <= PLOT_RANGE_ANGSTROM[1])
scale = np.nanmedian(improved.spectrum.flux[keep])

figure, (top, bottom) = plt.subplots(2, 1, figsize=(12, 6), sharex=True, gridspec_kw={"height_ratios": (3, 1)})
top.plot(wave[keep], improved.spectrum.flux[keep] / scale, color="black", alpha=0.55, label="Observed")
top.plot(wave[keep], incomplete.corrected.flux[keep] / scale, color="tab:red", label="Incomplete correction")
top.plot(wave[keep], improved.corrected.flux[keep] / scale, color="tab:blue", label="Improved correction")
top.set_ylabel("Normalized flux")
top.legend()
relative_change = 100 * (improved.corrected.flux[keep] / incomplete.corrected.flux[keep] - 1)
bottom.plot(wave[keep], relative_change, color="tab:purple")
bottom.axhline(0, color="black", linewidth=0.8)
bottom.set_ylabel("Change [%]")
bottom.set_xlabel("Air wavelength [Angstrom]")
figure.tight_layout()
plt.show()

## Build a fit health report

The telluric-depth slope below is a heuristic. After division, a positive trend of corrected normalized flux with absorption depth is consistent with overcorrection; a negative trend is consistent with undercorrection. Stellar lines, continuum error, and correlated noise can produce the same signal, so the slope must be inspected alongside the spectrum.

In [ ]:
def health_report(result):
    fit_mask = np.asarray(result.fit_mask, dtype=bool)
    continuum = np.asarray(result.continuum, dtype=float)
    corrected_norm = np.divide(
        result.corrected.flux, continuum, out=np.full_like(continuum, np.nan), where=continuum != 0
    )
    depth = 1 - np.asarray(result.transmission, dtype=float)
    valid = fit_mask & np.isfinite(corrected_norm) & np.isfinite(depth) & (depth > 0.005)
    slope = np.nan
    if np.count_nonzero(valid) >= 10:
        slope = float(np.polyfit(depth[valid], corrected_norm[valid], 1)[0])
    return {
        "optimizer_success": bool(result.success),
        "function_evaluations": int(result.nfev),
        "fit_pixels": int(np.count_nonzero(fit_mask)),
        "minimum_transmission": float(np.nanmin(result.transmission)),
        "saturated_pixel_fraction_T_lt_0.05": float(np.mean(result.transmission[fit_mask] < 0.05)),
        "parameters_at_bounds": result.parameter_bound_status or "none",
        "corrected_flux_vs_telluric_depth_slope": slope,
        **correction_summary(result),
    }

for label, fitted in (("Incomplete", incomplete), ("Improved", improved)):
    print(f"\n{label}")
    for key, value in health_report(fitted).items():
        print(f"  {key}: {value}")

## Symptom-to-action guide

| Symptom | Likely causes | Appropriate action |
|---|---|---|
| Adjacent positive/negative spikes | wavelength offset, wrong wavelength medium/frame, wrong LSF width | verify air/vacuum and topocentric frame; fit a bounded shift; refine the LSF |
| Corrected lines remain too deep | molecular column too low, missing species, incorrect atmosphere | verify species and fit windows; inspect atmosphere metadata; fit the molecular scale |
| Corrected lines become emission spikes | molecular column too high, source lines used as tellurics, saturated pixels | mask astrophysical features; restrict fit windows; exclude saturated cores |
| Broad oscillation or slope | continuum order too low/high, blaze residuals, order stitching | inspect each order; use a defensible local continuum; do not fit individual lines with the continuum |
| Parameter reaches a bound | inadequate bounds, degeneracy, missing physics, poor starting model | inspect the profile and residuals before widening bounds; do not automatically expand them |
| Fit changes strongly with segment size | insufficient overlap/margins, local continuum instability | inspect boundaries; increase margins or fit native orders; compare defensible segmentations |
| Optimizer succeeds but correction looks unchanged | no active lines/species in fit range, transmission near one, wrong units | print species scales, transmission minimum, fit mask, and wavelength range |
| Deep-line noise explodes | division by near-zero transmission | mask or flag saturated pixels; do not claim recovered flux where the atmosphere removed the information |

## Retain the auditable product

The corrected-only file is convenient for later analysis. The full product is the scientific record: it contains input and corrected flux, model, transmission, continuum, masks, parameters, and provenance. This compact HARPS crop has no per-pixel uncertainty column, so an absolute chi-square or calibrated corrected uncertainty cannot be inferred from it.

In [ ]:
product = Table.read(OUTPUT / "05_o2_fit_product.ecsv")
print("Product columns:", product.colnames)
print("Stored fit success:", product.meta.get("fit_success"))
print("Stored species scales:", product.meta.get("species_scales"))
print("Stored bound status:", product.meta.get("parameter_bound_status"))
print("Provenance field present:", "provenance_json" in product.meta)

## Minimum review before scientific use

Before accepting a correction, record and inspect: input wavelength unit/medium/frame; observation metadata and atmosphere source; fit and exclusion windows; active species; LSF family and bounds; optimizer convergence and bound status; transmission depth and saturation mask; residuals as a function of wavelength and telluric depth; sensitivity to defensible atmosphere/LSF alternatives; and at least one independent validation route.